## 1. Import Libraries and Setup

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from tqdm.auto import tqdm
from time import time

print(f"pandas version: {pd.__version__}")

## 2. Define Data Types

Specifying data types improves memory usage and parsing performance.

In [ ]:
# Data types for yellow taxi data
DTYPE = {
    "VendorID": "Int64",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "RatecodeID": "Int64",
    "store_and_fwd_flag": "string",
    "PULocationID": "Int64",
    "DOLocationID": "Int64",
    "payment_type": "Int64",
    "fare_amount": "float64",
    "extra": "float64",
    "mta_tax": "float64",
    "tip_amount": "float64",
    "tolls_amount": "float64",
    "improvement_surcharge": "float64",
    "total_amount": "float64",
    "congestion_surcharge": "float64"
}

# Columns to parse as datetime
PARSE_DATES = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]

print(f"Defined {len(DTYPE)} column types")
print(f"Date columns: {PARSE_DATES}")

## 3. Download Sample Data

Download a small sample of NYC Yellow Taxi data for exploration.

In [ ]:
# URL for yellow taxi data (January 2021)
URL = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz"

# Read only first 100 rows for quick exploration
df_sample = pd.read_csv(URL, dtype=DTYPE, parse_dates=PARSE_DATES, nrows=100)

print(f"Sample shape: {df_sample.shape}")
df_sample.head()

## 4. Explore Data Schema

In [ ]:
# Check data types
df_sample.dtypes

In [ ]:
# Basic statistics
df_sample.describe()

In [ ]:
# Check for null values
df_sample.isnull().sum()

## 5. Generate SQL Schema

Use pandas to generate the CREATE TABLE statement for PostgreSQL.

In [ ]:
# Generate CREATE TABLE statement
print(pd.io.sql.get_schema(df_sample, name="yellow_taxi_data"))

## 6. Connect to PostgreSQL

Create a connection to the PostgreSQL database running in Docker.

In [ ]:
# Database connection parameters
PG_HOST = "localhost"
PG_PORT = 5432
PG_USER = "root"
PG_PASS = "root"
PG_DB = "ny_taxi"

# Create SQLAlchemy engine
connection_string = f"postgresql://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
engine = create_engine(connection_string)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute("SELECT 1")
        print("✅ Connected to PostgreSQL successfully!")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 7. Ingest Data in Chunks

For large datasets, read and insert data in chunks to manage memory.

In [ ]:
# Read data in chunks
CHUNKSIZE = 100000

df_iter = pd.read_csv(
    URL,
    dtype=DTYPE,
    parse_dates=PARSE_DATES,
    iterator=True,
    chunksize=CHUNKSIZE
)

# Get first chunk
df_chunk = next(df_iter)
print(f"Chunk shape: {df_chunk.shape}")
print(f"Memory usage: {df_chunk.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Create table and insert first chunk
TABLE_NAME = "yellow_taxi_data"

t_start = time()
df_chunk.to_sql(name=TABLE_NAME, con=engine, if_exists="replace", index=False)
t_end = time()

print(f"✅ Created table '{TABLE_NAME}' and inserted {len(df_chunk):,} rows")
print(f"⏱️ Time: {t_end - t_start:.2f} seconds")

In [ ]:
# Insert remaining chunks (run only if you want full data)
# Warning: This may take several minutes

total_rows = len(df_chunk)

for df_chunk in tqdm(df_iter, desc="Ingesting chunks"):
    t_start = time()
    df_chunk.to_sql(name=TABLE_NAME, con=engine, if_exists="append", index=False)
    t_end = time()
    total_rows += len(df_chunk)

print(f"\n✅ Total rows ingested: {total_rows:,}")

## 8. Load Taxi Zones Lookup Table

In [ ]:
# Download and load taxi zones
ZONES_URL = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv"

df_zones = pd.read_csv(ZONES_URL)
print(f"Zones shape: {df_zones.shape}")
df_zones.head()

In [ ]:
# Insert zones into database
df_zones.to_sql(name="taxi_zones", con=engine, if_exists="replace", index=False)
print(f"✅ Inserted {len(df_zones)} zones into 'taxi_zones' table")

## 9. Run SQL Queries from Python

In [ ]:
# Count total trips
query = """
SELECT COUNT(*) as total_trips
FROM yellow_taxi_data;
"""

pd.read_sql(query, engine)

In [ ]:
# Top pickup locations
query = """
SELECT 
    z."Zone" as pickup_zone,
    COUNT(*) as trip_count
FROM yellow_taxi_data t
JOIN taxi_zones z ON t."PULocationID" = z."LocationID"
GROUP BY z."Zone"
ORDER BY trip_count DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

In [ ]:
# Average trip statistics by day
query = """
SELECT 
    DATE(tpep_pickup_datetime) as pickup_date,
    COUNT(*) as trips,
    ROUND(AVG(trip_distance)::numeric, 2) as avg_distance,
    ROUND(AVG(total_amount)::numeric, 2) as avg_total
FROM yellow_taxi_data
GROUP BY DATE(tpep_pickup_datetime)
ORDER BY pickup_date
LIMIT 10;
"""

pd.read_sql(query, engine)